# MSR-VTT pooling

Public adaptation of the historical Kaggle workflow. Python modules are byte-identical to captured sources; this notebook replaces embedded source/data payloads with verified files. It has no execution outputs and has not been rerun as a GPU job. Read docs/protocols/kaggle_research_sources.md before running. Attach authorized inputs separately; no test-based model selection.

In [ ]:
# Cell 1 — Không cài đè torch/CUDA/numpy/OpenCV của Kaggle.
import importlib.util
import json
import os
import subprocess
import sys
from pathlib import Path

base = Path(os.environ.get("MSRVTT_WORKSPACE", "/kaggle/working/msrvtt_attention"))
CODE_DIR = base / "code"
CODE_DIR.mkdir(parents=True, exist_ok=True)
if str(CODE_DIR) not in sys.path:
    sys.path.insert(0, str(CODE_DIR))
for name in ("torch", "numpy", "cv2", "PIL"):
    if importlib.util.find_spec(name) is None:
        raise RuntimeError(f"Thiếu {name}: chọn image Python/GPU chuẩn của Kaggle.")
packages = {"ftfy": "ftfy==6.3.1", "regex": "regex==2026.9.29", "tqdm": "tqdm==4.70.1"}
missing = [p for m, p in packages.items() if importlib.util.find_spec(m) is None]
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet",
                           "--disable-pip-version-check", *missing])
torch = importlib.import_module("torch")
print("Python", sys.version.split()[0], "torch", torch.__version__,
      "CUDA", torch.cuda.is_available())


In [ ]:
# Cấu hình lấy từ lượt chạy đã lưu; các đường dẫn dữ liệu được để trống.
CONFIG = {'INPUT_ROOT': '/kaggle/input',
 'WORK_DIR': '',
 'VIDEO_ROOT': '',
 'TEST_CSV': '',
 'TRAIN_CSV': '',
 'CAPTIONS_JSON': '',
 'CHECKPOINT_PATH': '',
 'CHECKPOINT_SHA256': '3d7e2ba508986c7347380767233b017d4ac4c9fabc135f3457de8625dc833dca',
 'DOWNLOAD_CHECKPOINT': True,
 'HF_REVISION': 'de15dc5686a8628d051dae0a85b80d34ed690c68',
 'HF_FILENAME': 'ckpts/ckpt_msrvtt_retrieval_looseType/pytorch_model.bin.4',
 'RUN_TRAIN': False,
 'SELECTION_POLICY': 'fixed_epochs',
 'DEV_VAL_SIZE': 900,
 'VAL_CSV': '',
 'VAL_SEEN_BY_ENCODER': True,
 'VAL_PROVENANCE': '',
 'ALLOW_SEEN_ENCODER_DEV': False,
 'EPOCHS': 5,
 'TRAIN_BATCH': 32,
 'LR': 0.001,
 'WEIGHT_DECAY': 0.0001,
 'TRAIN_LOGIT_SCALE': 20.0,
 'RESUME': True,
 'MAX_FRAMES': 12,
 'MAX_WORDS': 32,
 'PRECISION': 'fp32',
 'ENCODER_BATCH': 4,
 'TEXT_BATCH': 1,
 'QUERY_CHUNK': 32,
 'VIDEO_CHUNK': 32,
 'TAU': 0.05,
 'ALPHA': 0.25,
 'SEED': 42,
 'BOOTSTRAP_SAMPLES': 2000,
 'EXPECTED_TEST_QUERIES': 1000,
 'EXPECTED_TEST_VIDEOS': 1000,
 'EXPECTED_BASELINE_R1': 41.0,
 'EXPECTED_R1_TOL_PP': 1e-08,
 'ATOL': 1e-05,
 'RTOL': 1e-05,
 'BASELINE_REFERENCE_NPZ': '',
 'BASELINE_REFERENCE_SHA256': '429403c137f57f98ff93d7f3fcf121e32a5c3b88f0b60688c784a2028eaa15ba',
 'REFERENCE_IS_LOGITS': True,
 'REQUIRE_EXTERNAL_REFERENCE': True,
 'ALLOW_UNVERIFIED_EXPLORATORY': False,
 'ALLOW_CPU_DEBUG': False,
 'MIN_FREE_DISK_GIB': 2.0,
 'RESOURCE_PROBE_VIDEOS': 8}
CONFIG["WORK_DIR"] = str(base / "public_pooling_run")


In [ ]:
# Thay payload nhúng bằng mã công khai có SHA256; input dữ liệu vẫn được gắn riêng.
import importlib.util

PUBLIC_CODE = Path("/kaggle/working/temporal-hoi-video-text-matching/scripts/kaggle/msrvtt")
_spec = importlib.util.spec_from_file_location("prepare_public_sources", PUBLIC_CODE / "prepare_sources.py")
_prepare = importlib.util.module_from_spec(_spec)
_spec.loader.exec_module(_prepare)
_prepare.restore_sources(PUBLIC_CODE, CODE_DIR)
for _module in ("data_pipeline", "retrieval_core", "workflow", "resources", "bpe_bootstrap", "b3_probe_runtime", "fulltrain_runtime", "b3_eval_runtime"):
    sys.modules.pop(_module, None)
VENDOR_DIR = CODE_DIR / "vendor"
importlib.import_module("bpe_bootstrap").ensure_bpe(VENDOR_DIR / "bpe_simple_vocab_16e6.txt.gz", CONFIG["INPUT_ROOT"])
_prepare.capture_stage(PUBLIC_CODE, CODE_DIR, Path(CONFIG["WORK_DIR"]), "pooling", Path(CONFIG["INPUT_ROOT"]))
_work = Path(CONFIG["WORK_DIR"])


In [ ]:
# Cell 4 — Self-test CPU tổng hợp, không phải benchmark MSR-VTT.
import importlib
import unittest

importlib.invalidate_caches()
for module in ("workflow", "resources", "test_core", "test_pipeline", "data_pipeline", "retrieval_core"):
    sys.modules.pop(module, None)
test_core = importlib.import_module("test_core")
test_pipeline = importlib.import_module("test_pipeline")

suite = unittest.TestSuite([
    unittest.defaultTestLoader.loadTestsFromModule(test_core),
    unittest.defaultTestLoader.loadTestsFromModule(test_pipeline),
])
result = unittest.TextTestRunner(verbosity=1).run(suite)
if not result.wasSuccessful():
    raise RuntimeError("Self-test lỗi; giữ log và dừng trước workload dài.")


In [ ]:
# Cell 5 — Kiểm input, khóa thí nghiệm.
from workflow import benchmark, cache_split, load_runtime, preflight, train_and_benchmark

STATE = preflight(CONFIG)


In [ ]:
# Cell 6 — Nạp encoder epoch4 FP32 và warm-up.
STATE = load_runtime(STATE, VENDOR_DIR)
print(STATE["model_info"])


In [ ]:
# Cell 7 — Đo mẫu nhỏ trước lượt dài.
from resources import resource_probe

RESOURCE_REPORT = resource_probe(STATE)
print(json.dumps(RESOURCE_REPORT, ensure_ascii=False, indent=2))


In [ ]:
# Cell 8 — Cache frame/text và metadata.
TEST_CACHE = cache_split(STATE, "test")
print("frames", tuple(TEST_CACHE["frames"].shape), "text", tuple(TEST_CACHE["text"].shape))


In [ ]:
# Cell 9 — Baseline gate, B0/B1/B2 trên toàn gallery.
REPORT, BASE_RESULTS = benchmark(STATE, TEST_CACHE)
print(json.dumps(REPORT["baseline_gate"], ensure_ascii=False, indent=2))
for variant, row in REPORT["variants"].items():
    for direction in ("text_to_video", "video_to_text"):
        print(variant, direction, row["internal"][direction])


In [ ]:
# Cell 10 — Head train, test chỉ chấm sau khi học xong.
TRAINED_REPORT = train_and_benchmark(STATE, TEST_CACHE, REPORT, BASE_RESULTS)
if TRAINED_REPORT is not None:
    print("Epoch dùng:", TRAINED_REPORT["best_epoch"])
    print("Trạng thái:", TRAINED_REPORT["development_status"])
    for direction in ("text_to_video", "video_to_text"):
        print("B3", direction, TRAINED_REPORT["internal"][direction])


In [ ]:
# Cell 11 — ZIP báo cáo, giữ cache/encoder ở Output.
import zipfile

result_dir = Path(CONFIG["WORK_DIR"])
result_zip = result_dir.parent / (result_dir.name + "_results.zip")
with zipfile.ZipFile(result_zip, "w", zipfile.ZIP_DEFLATED) as archive:
    for item in sorted(result_dir.rglob("*")):
        parts = item.relative_to(result_dir).parts
        if item.is_file() and not ({"cache", "checkpoint"} & set(parts)):
            archive.write(item, item.relative_to(result_dir))
print("Reports:", result_dir)
print("Download:", result_zip, "MiB", round(result_zip.stat().st_size / 2**20, 2))
